# A4 · `explain_hold`: Parts 1 to 4, executed

The traces quoted in `TOOL.md`, `MCP.md` and `HARNESSES.md`. Ledgerline is the course kit's `fin_payables` module with my capstone's records (`capstone_seed.py`); the sandbox date is Monday 26 October 2026.
- **Parts 1, 2 and 4** run here, live against the sandbox. They need no model.
- **Part 3** made live model calls on my own computer (`run_part3.py`). The cells below read what it saved in `logs/`.

## Part 1 · Load the tool next to the shipped ones, and lint it

In [1]:
from a4_setup import load_tool
from ledgerline.course.l4 import lint_tool, validate_args
from ledgerline.toolbox import SPECS, ORDER
name = load_tool()
print(len(ORDER), "shipped tools +", name)
print("lint_tool:", lint_tool(name))
print("validate_args on a number:", validate_args(name, {"invoice_no": 4471}))

15 shipped tools + explain_hold
lint_tool: []
validate_args on a number: ["invoice_no: 4471 is not of type 'string'"]


### Every case, with the bytes from Ledgerline and the bytes the tool returns

In [2]:
%run part1_run.py

lint_tool: []
jsonschema on {'invoice_no': 4471}: ["invoice_no: 4471 is not of type 'string'"]

## happy path: Voltway Cells' 4471
args: {"invoice_no": "4471", "supplier": "Voltway Cells"}
Ledgerline calls (7): ap_inv_list, ap_inv_get, ap_inv_list, usr_get, usr_get, usr_get, usr_get · 2,679 bytes in envelopes -> 854 bytes returned · output-schema violations: 0
{
 "ok": true,
 "supplier": "Voltway Cells Pvt. Ltd.",
 "ap_no": "AP/2026/03390",
 "status": "on hold",
 "invoice_no": "4471",
 "amount": 28308672.0,
 "due_date": "2026-11-16",
 "payment": {
  "last_run": "2026-11-13",
  "cutoff": "2026-11-12",
  "this_friday": "2026-10-30",
  "payable_this_friday": false,
  "why_not": "on hold; nothing is paid while an invoice is on hold"
 },
 "holds": [
  {
   "code": "H-QTY",
   "what_is_wrong": "18,000 billed, 17,700 accepted",
   "on_hold_since": "2026-10-05",
   "may_release": [
    {
     "name": "Ganesh Pawar",
     "role": "stores"
    },
    {
     "name": "Vikram Nair",
     "role": "f

### The tests

In [3]:
%run tests/test_explain_hold.py

PASS test_a_person_who_really_left_is_still_reported
PASS test_business_answer_two_suppliers_same_number
PASS test_busy_usr_get_is_an_outage_not_a_policy_change
PASS test_clean_result_not_the_envelope
PASS test_happy_path_two_holds_who_and_when
PASS test_ledgerline_busy_retried_then_given_up
PASS test_lost_reply_gives_a_next_step_this_server_can_follow
PASS test_refused_at_the_door_missing_argument
PASS test_refused_at_the_door_wrong_type


PASS test_refused_by_ledgerline_unknown_ap_number
PASS test_unknown_invoice_number
11 passed


## Part 2 · The raw MCP wire

In [4]:
%run part2_wire.py

-> {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18", "capabilities": {}, "clientInfo": {"name": "a4-part2", "version": "1.0"}}}

<- {"jsonrpc": "2.0", "id": 1, "result": {"protocolVersion": "2025-06-18", "capabilities": {"logging": {}, "prompts": {"listChanged": true}, "resources": {"subscribe": false, "listChanged": true}, "tools": {"listChanged": true}}, "serverInfo": {"name": "northbridge-ap", "version": "4.1.8"}, "instructions": "Northbridge Cycles' Ledgerline ERP, as PSHAH. One tool per business activity."}}

-> {"jsonrpc": "2.0", "method": "notifications/initialized"}

-> {"jsonrpc": "2.0", "id": 2, "method": "tools/list"}

<- {"jsonrpc": "2.0", "id": 2, "result": {"tools": [{"_meta": {"ledgerline/category": "Look up", "ledgerline/writes": false, "ledgerline/services": ["ap_inv_list", "ap_inv_get", "usr_get"], "ledgerline/policy": {"ledgerline_reply_timeout_s": 30, "on_busy_or_lost_reply": "retry the read, at most 2 more times", "afte

## Part 3 · Our harness: the context at every model call (recorded live by `part3_ours.py`)

In [5]:
import json, pathlib
print(pathlib.Path('logs/part3_ours_context.txt').read_text(encoding='utf-8'))
r = json.load(open('logs/part3_ours.json', encoding='utf-8'))
for t in r['turns']:
    print('EMMA:', t['user'])
    for c in t['cards']:
        print('   call', c['tool'], json.dumps(c['args']))
    print('   REPLY:', t['reply'])
    print(f"   model calls {t['model_calls']} · tokens in {t['tokens_in']:,} · out {t['tokens_out']:,} · {t['seconds']} s")
print('tool definitions:', r['tool_definition_tokens'], 'tokens of the first call\'s', r['first_call_prompt_tokens'])

-- model call 1 of 4: prompt_tokens 434 --
   +    216  system prompt                    
   +    195  tool definitions (1)             
   +     23  user message                     Why is Voltway's invoice 4471 on hold, and can we pay it Friday?
     = 434 tokens in the context (matches prompt_tokens)
     model wrote (28 tokens): explain_hold({"supplier": "Voltway", "invoice_no": "4471"})
-- model call 2 of 4: prompt_tokens 756 --
   +    141  tool call                        [{"explain_hold": {"supplier": "Voltway", "invoice_no": "4471"}}]
   +    181  tool result                      {"ok": true, "matches": [{"ap_no": "AP/2026/03409", "supplier": "Voltw
     = 756 tokens in the context (matches prompt_tokens)
     model wrote (92 tokens): 'Two suppliers have issued an invoice numbered 4471:\n- Voltway Paints & Coatings (invoice date 20 Oct 2026, ₹2,'
-- model call 3 of 4: prompt_tokens 858 --
   +    102  assistant reply (+ the user's next message) Two suppliers have issued an inv

## Part 3 · Claude Code, headless (recorded live by `run_part3.py`)

In [6]:
import glob, subprocess, sys
files = sorted(glob.glob('logs/claude_turn*.jsonl'))
print(subprocess.run([sys.executable, 'part3_claude_summary.py', *files], capture_output=True, text=True).stdout)

=== logs/claude_turn1.jsonl · model claude-opus-5-5 · MCP [{'name': 'northbridge-ap', 'status': 'connected', 'source': 'dynamic'}] · tools offered 20
  CALL   ToolSearch {"query": "select:mcp__northbridge-ap__explain_hold", "max_results": 1}
  RESULT  
  CALL   mcp__northbridge-ap__explain_hold {"invoice_no": "4471", "supplier": "Voltway"}
  RESULT  {"ok":true,"matches":[{"ap_no":"AP/2026/03409","supplier":"Voltway Paints & Coatings","invoice_date":"2026-10-20","amount":217120,"status":"clear to pay"},{"ap_no":"AP/2026/03390","supplier":"Voltway Cells Pvt. Ltd.","invoice_date":"2026-10-02","amount":28308672,"status":"on hold"}],"next_step":"2 suppliers used invoice number 4471. Ask the user which: Voltway Paints & Coatings or Voltway Cells Pv
  TEXT   Two suppliers have used invoice number 4471. Only Voltway Cells' invoice is on hold, so I'll look that one up.
  CALL   mcp__northbridge-ap__explain_hold {"invoice_no": "4471", "supplier": "Voltway Cells"}
  RESULT  {"ok":true,"supplier":

## Part 4 · Production grade: seven criteria, measured live

In [7]:
%run production.py


## 1 schema  (1.8 s)
{
 "results_checked": 724,
 "violations": 0,
 "both_schemas_valid": true,
 "examples": []
}



## 2 reliability  (4.7 s)
{
 "calls": 1903,
 "outside_contract": 0,
 "share": 0.0,
 "kinds": {},
 "examples": {}
}



## 4 latency  (0.2 s)
{
 "calls": 800,
 "tool_own_ms": {
  "p50": 0.096,
  "p95": 0.202,
  "p99": 0.235
 },
 "with_ledgerline_ms": {
  "p50": 0.274,
  "p95": 0.477,
  "p99": 0.577
 },
 "limits_ms": {
  "p50": 50,
  "p95": 100,
  "p99": 200
 }
}

## 5 payload  (0.1 s)
{
 "largest_real_result_bytes": 854,
 "limit_bytes": 4096,
 "same_number_from_many_suppliers": {
  "10": {
   "bytes": 2198,
   "rows": 12,
   "says_how_many": false
  },
  "50": {
   "bytes": 9238,
   "rows": 52,
   "says_how_many": false
  },
  "200": {
   "bytes": 35639,
   "rows": 202,
   "says_how_many": false
  }
 }
}

## 7 timeouts  (0.0 s)
{
 "tests": 6,
 "passed": 6,
 "failed": [],
 "failed_results": []
}



## 8 regression  (0.2 s)
{
 "tests_passed": 11,
 "tests_run": 11,
 "all_pass": true,
 "failure": ""
}



## 9 mcp compliance  (1.6 s)
{
 "checks": 10,
 "passed": 10,
 "detail": {
  "initialize agrees 2025-06-18": true,
  "ping answers {}": true,
  "tools/list: one tool with name, description, inputSchema": true,
  "inputSchema and outputSchema are valid JSON Schema": true,
  "a good call: isError false, structuredContent fits outputSchema": true,
  "content text and structuredContent say the same": true,
  "a failed call: isError true, with the error contract": true,
  "a wrong argument type: refused, not crashed": true,
  "an unknown method: JSON-RPC error -32601": true,
  "an unknown tool: refused": true
 }
}

written logs/production.json


### The failure found while measuring, and the same faults now

In [8]:
import json
found = json.load(open('logs/production_before_fix.json', encoding='utf-8'))['criteria']['7 timeouts']
print('timeouts when first measured:', found['passed'], 'of', found['tests'])
for f in found['failed']:
    print('  ', f['service'], f['fault'], 'x', f['times'], '-', f['why'][:120])
from a4_setup import sandbox
for svc, mode in [('usr_get', 'error_before')]:
    erp, tb = sandbox(user='ECLARKE', faults={svc: {'mode': mode, 'times': 3}})
    print('now:', svc, mode, 'x3 ->', tb.call('explain_hold', {'invoice_no': 'AP/2026/03390'}))

timeouts when first measured: 5 of 6
   usr_get error_before x 3 - answered as if nothing failed: AP policy 4.1 is out of date: GPAWAR is no longer in Ledgerline. Ask Vikram Nair who rele
now: usr_get error_before x3 -> {'ok': False, 'error': 'Ledgerline refused usr_get: Server busy, try later (ERR-SYS-0503).', 'next_step': 'Ledgerline is busy. The tool already retried; try again in a minute, or tell the user.', 'ledgerline_code': 'ERR-SYS-0503'}
